# chess-rl-bench — L3 training on Kaggle

Clones the portfolio repo, trains the L3 perceptron at scale, evaluates it against a reference policy, and prints a summary.

**Internet must be enabled** for the clone + `pip install` (already set in the kernel metadata).

Run the tiny validation config first (see the config cell), confirm it works, then scale up. The trained model is written to `/kaggle/working/out` and is downloadable from the notebook output.

In [ ]:
# 1. Fetch the code and install dependencies
!git clone --depth 1 https://github.com/cklt-90/Chess-dumbAI-benchmarking.git repo
!pip install -q python-chess numpy
!ls repo/bench/kaggle

## Run configuration

Edit the values below. Suggested **first run (validation)**: `MODE='both'`, `GAMES=20`, `POSITIONS=300`, `SEARCH_DEPTH=3`, `EVAL_GAMES=20` — confirms the clone, install, training, and eval all work on Kaggle.

**Scale run**: `GAMES=4000`, `POSITIONS=24000` (depth-3 labelling ≈ 1.6 h on CPU), `EVAL_GAMES=40`.

Labelling cost scales steeply with depth (measured): ~0.04 s/pos (d2), ~0.24 s/pos (d3), ~4.5 s/pos (d4). Keep depth 2–3 at scale; depth 4+ is not viable. Self-play training and the eval are cheap by comparison.

In [ ]:
import subprocess

MODE = 'both'
GAMES = 20
POSITIONS = 300
SEARCH_DEPTH = 3
EVAL_AGAINST = 'material'
EVAL_GAMES = 20
SEED = 7
# Scale run: GAMES=4000, POSITIONS=24000 (~1.6h on CPU), EVAL_GAMES=40

cmd = (
    f'python repo/bench/kaggle/kaggle_l3_train.py '
    f'--mode {MODE} --games {GAMES} --positions {POSITIONS} '
    f'--search-depth {SEARCH_DEPTH} --eval-against {EVAL_AGAINST} '
    f'--eval-games {EVAL_GAMES} --seed {SEED} --out-dir /kaggle/working/out'
)
print('Running:', cmd)
subprocess.run(cmd, shell=True, check=True)

In [ ]:
import pathlib

out = pathlib.Path('/kaggle/working/out')
if out.exists():
    print(out.joinpath('summary.md').read_text())
    print('\nArtifacts written to /kaggle/working/out (downloadable from the notebook output):')
    for p in sorted(out.glob('*.json')):
        print(' -', p.name, f'({p.stat().st_size/1024:.0f} KB)')
else:
    print('No output directory found - the training step may have failed.')